Fiche technique

| DONNÉE             | SÉLECTEUR       | CONTENU OU ATTRIBUT ? |
| :----------------- | :-------------- | :-------------------- |
| titre              | `h3 a`          | Attribut (title)      |
| prix               | `p.price_color` | Contenu               |
| disponibilité      | `p.instock`     | Contenu               |
| note               | `p.star-rating` | Attribut (class)      |
| lien vers la fiche | `h3 a`          | Attribut (href)       |

## 1. Apprentissage

In [17]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (ton.email@exemple.fr)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status :", r.status_code)
print("longueur :", len(r.text), "caractères")
print("début :", r.text[:60])

status : 200
longueur : 50449 caractères
début : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [18]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (ton.email@exemple.fr)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status :", r.status_code)
print("longueur :", len(r.text), "caractères")
print("début :", r.text[:60])

status : 200
longueur : 50449 caractères
début : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [19]:
r = requests.get(URL, headers=ENTETES, timeout=10)

print("encodage deviné par requests :", r.encoding)
print("encodage réel de la page :", r.apparent_encoding)
print("Le prix :", r.text[r.text.find("price_color") : r.text.find("price_color") + 80])

encodage deviné par requests : ISO-8859-1
encodage réel de la page : utf-8
Le prix : price_color">Â£51.77</p>
    

<p class="instock availability">
    <i class="ic


In [21]:
r.encoding = "utf-8"

print("Le prix :", r.text[r.text.find("price_color"):r.text.find("price_color") + 80])

print("le prix est present :", "£51.77" in r.text)
print("le titre est present :", "A Light in the Attic" in r.text)

Le prix : price_color">£51.77</p>
    

<p class="instock availability">
    <i class="ico
le prix est present : True
le titre est present : True


In [22]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvées :", len(fiches))

fiches trouvées : 20


In [23]:
fiche = fiches[0]

# Un CONTENU d'une balise
prix = fiche.select_one("p.price_color").get_text(strip=True)

# Un ATTRIBUT
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix :", prix)
print("titre :", titre)
print("href :", href)


prix : £51.77
titre : A Light in the Attic
href : a-light-in-the-attic_1000/index.html


In [24]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

classes : ['star-rating', 'Three']
note : 3


In [25]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [26]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]

    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': '£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [27]:
suivant = soup.select_one("li.next a")

print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [28]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1

        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"page {page} : status {r.status_code}, on s'arrête")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]

        print(f"  page {page} : {len(fiches)} livres (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres

livres = collecter_toutes_les_pages(
    "https://books.toscrape.com/catalogue/page-1.html",
    pages_max=3
)
print("Collecte :", len(livres), "livres")

  page 1 : 20 livres (total 20)
  page 2 : 20 livres (total 40)
  page 3 : 20 livres (total 60)
Collecte : 60 livres


In [29]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # Le tableau : une entrée par ligne
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # La catégorie : troisième élément du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }

url_test = livres[0]["url"]
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [30]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0

print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


## Challenge — collecter un catalogue sur deux niveaux

In [31]:
import pandas as pd

URL_CATEGORIE = "https://books.toscrape.com/catalogue/category/books/poetry_23/index.html"

r = requests.get(URL_CATEGORIE, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

soup = BeautifulSoup(r.text, "html.parser")
fiches = soup.select("article.product_pod")

livres_poetry = [lire_fiche(f, URL_CATEGORIE) for f in fiches]

df = pd.DataFrame(livres_poetry)

print("Nombre de livres :", len(df))
display(df)

Nombre de livres : 19


,titre,prix,dispo,note,url
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,The Black Maria,£52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...
2,Shakespeare's Sonnets,£20.66,In stock,4,https://books.toscrape.com/catalogue/shakespea...
3,Olio,£23.88,In stock,1,https://books.toscrape.com/catalogue/olio_984/...
4,You can't bury them all: Poems,£33.63,In stock,2,https://books.toscrape.com/catalogue/you-cant-...
5,Slow States of Collapse: Poems,£57.31,In stock,3,https://books.toscrape.com/catalogue/slow-stat...
6,Untitled Collection: Sabbath Poems 2014,£14.27,In stock,4,https://books.toscrape.com/catalogue/untitled-...
7,Poems That Make Grown Women Cry,£14.19,In stock,4,https://books.toscrape.com/catalogue/poems-tha...
8,Night Sky with Exit Wounds,£41.05,In stock,1,https://books.toscrape.com/catalogue/night-sky...
9,salt.,£46.78,In stock,4,https://books.toscrape.com/catalogue/salt_731/...


In [32]:
livres_poetry = collecter_toutes_les_pages(URL_CATEGORIE, pages_max=3)

print("Nombre total :", len(livres_poetry))

  page 1 : 19 livres (total 19)
Nombre total : 19


In [33]:
for livre in livres_poetry:
    print("Titre :", livre["titre"])
    print("Catégorie :", livre["categorie"])
    print("Stock :", livre["stock"])
    print("UPC :", livre["upc"])
    print("---")

Titre : A Light in the Attic


KeyError: 'categorie'

In [ ]:
df = pd.DataFrame(livres_poetry)

df["prix_gbp"] = df["prix"].str.extract(r"([\d.]+)").astype(float)
df["note"] = df["note"].astype(int)
df["stock"] = df["stock"].astype(int)

df = df[[
    "titre", "categorie", "prix_gbp",
    "note", "stock", "upc", "url"
]]

print("Nombre de livres :", len(df))
print("Colonnes :", df.columns.tolist())

display(df)

Nombre de livres : 19
Colonnes : ['titre', 'categorie', 'prix_gbp', 'note', 'stock', 'upc', 'url']


,titre,categorie,prix_gbp,note,stock,upc,url
0,A Light in the Attic,Poetry,51.77,3,22,a897fe39b1053632,https://books.toscrape.com/catalogue/a-light-i...
1,The Black Maria,Poetry,52.15,1,19,1dfe412b8ac00530,https://books.toscrape.com/catalogue/the-black...
2,Shakespeare's Sonnets,Poetry,20.66,4,19,30a7f60cd76ca58c,https://books.toscrape.com/catalogue/shakespea...
3,Olio,Poetry,23.88,1,19,feb7cc7701ecf901,https://books.toscrape.com/catalogue/olio_984/...
4,You can't bury them all: Poems,Poetry,33.63,2,17,55f9da0c5eea2e10,https://books.toscrape.com/catalogue/you-cant-...
5,Slow States of Collapse: Poems,Poetry,57.31,3,17,b4fd5943413e089a,https://books.toscrape.com/catalogue/slow-stat...
6,Untitled Collection: Sabbath Poems 2014,Poetry,14.27,4,16,657fe5ead67a7767,https://books.toscrape.com/catalogue/untitled-...
7,Poems That Make Grown Women Cry,Poetry,14.19,4,15,31001a1210623bd4,https://books.toscrape.com/catalogue/poems-tha...
8,Night Sky with Exit Wounds,Poetry,41.05,1,15,cfeb1e1e0e695b58,https://books.toscrape.com/catalogue/night-sky...
9,salt.,Poetry,46.78,4,14,86cbddb61ea78bb7,https://books.toscrape.com/catalogue/salt_731/...


In [ ]:
print("Nombre de livres :", len(df))
print("Stock total :", df["stock"].sum())

Nombre de livres : 19
Stock total : 235


In [ ]:
prix_moyen = df["prix_gbp"].mean()
prix_max = df["prix_gbp"].max()

livre_plus_cher = df[df["prix_gbp"] == prix_max]

print("Prix moyen :", round(prix_moyen, 2), "£")
print("Prix maximum :", prix_max, "£")
print("Livre le plus cher :")
display(livre_plus_cher[["titre", "prix_gbp"]])

Prix moyen : 35.97 £
Prix maximum : 57.31 £
Livre le plus cher :


,titre,prix_gbp
5,Slow States of Collapse: Poems,57.31


In [ ]:
repartition_notes = df["note"].value_counts().sort_index()

print("Répartition des notes :")
print(repartition_notes)


Répartition des notes :
note
1    3
2    2
3    2
4    6
5    6
Name: count, dtype: int64


note
1    3
2    2
3    2
4    6
5    6
Name: count, dtype: int64

In [ ]:
print(df["categorie"].value_counts())


categorie
Poetry    19
Name: count, dtype: int64


In [ ]:
df.to_csv("../data/poetry.csv", index=False)